In [29]:
import numpy as np

W1 = {
    0: {2, 4},
    1: {3, 4},
    2: {1},
    3: {0, 1, 4},
    4: set()}

W2 = {
    0: {1},
    1: {2},
    2: {0},
    3: {4},
    4: {5},
    5: {3}}

In [30]:
def surf_step(web):
    
    # Input: Et netværk som dictionary og en start side
    # Output: Sandsynlighedsfordeling som dictionary for næste hjemmeside
    
    distribution=dict() # laver en tom dictionary til at gemme sandsynlighedsfordelingen

    for page, links in web.items(): # for hver hjemmeside og dens links i web-dictionaryen
        if links == set(): # hvis hjemmesiden ikke har nogen links, så skal den have en ligelig fordeling over alle sider
            distribution[page] = [1/len(web) for i in range(len(web))] # fordel sandsynligheden ligeligt over alle sider fx laver den [0, 0, 0, 0, 0] -> [1/5, 1/5, 1/5, 1/5, 1/5]
        else: # hvis hjemmesiden har links, så skal den fordele sandsynligheden ligeligt over de sider den linker til
            distribution[page] = [0 for i in range(len(web))] # starter med at lave [0, 0, 0, 0, 0] for hver hjemmeside
            for link in links:
                distribution[page][link] += 1/len(links) # her taget den fx [0, 1, 0, 1, 0] og laver den til [0, 1/2, 0, 1/2, 0] hvis der er 2 links

    
    return distribution

In [31]:
def modified_link_matrix(web, pagelist=None, d=0.85):

    # Input: web (dictionary), pagelist (liste over nøgler), d (dæmpningsfaktor)
    # Output: d*A^T + (1-d)*E/N
    
    N=len(web)
    E_N = np.ones((N, N))
    distri = surf_step(web)
    L = np.array([i for i in distri.values()])

    # A: NxN numpy array, hvor række j har ikke-nul elementer i søjler, som side j linker til.
    # Hvis side j ikke linker til nogen, får alle elementer i række j værdien 1/N.
    # E: np.ones([N,N])

    M_d = (((1-d)/N) * E_N) + d*(L.T)
    return M_d

# Opgave 24

In [32]:
def eigenvector_PageRank(web,d=0.85):
        # Input: web er en ordbog over websider og links.
        # d er dæmpningen
        # Output: En ordbog med de samme nøgler som web og værdierne er PageRank for nøglerne

        ranking = dict()
        M = modified_link_matrix(web, d=d)
        eigenvalues, eigenvects = np.linalg.eig(M)

        idx = np.argmax(np.abs(eigenvalues))
        v = eigenvects[:,idx]/np.sum(eigenvects[:,idx])

        for i, page in enumerate(web.keys()):
                ranking[page] = v[i].real  # .real fjerner den imaginære del (0j)

        return ranking

eigenvector_PageRank(W1)

{0: 0.1329106202905259,
 1: 0.24908976517485165,
 2: 0.13668134692273656,
 3: 0.18605748349857465,
 4: 0.2952607841133112}